Name: **Areeb Ishaq**

Reg No: **FA23-BAI-063**

----

# Lab 02 Data Acquisition for NLP

Task 1: Multi-Page PDF Extraction with Word Frequency Analysis

In [3]:
import PyPDF2
import re

print("Part A")
file = open("Wahab Dawakhana.pdf", "rb")
reader = PyPDF2.PdfReader(file)
pages = []
for page in reader.pages:
    text = page.extract_text()
    if text is None:
        text = ""
    pages.append(text)
print("Total pages extracted:", len(pages))

print("\nPart B")
full_text = " ".join(pages)
full_text = re.sub(r"[^a-zA-Z\s]", "", full_text)
full_text = full_text.lower()
tokens = full_text.split()
print("Total token count:", len(tokens))

print("\nPart C")
stopwords = {
    "the", "is", "a", "an", "and", "of", "to", "in", "on", "for",
    "with", "that", "this", "it", "as", "at", "by", "from", "or", "be"
}
word_freq = {}
for token in tokens:
    if token not in stopwords:
        if token in word_freq:
            word_freq[token] += 1
        else:
            word_freq[token] = 1
sorted_words = sorted(word_freq.items(), key=lambda item: item[1], reverse=True)
print("Top 10 most frequent words:")
for word, count in sorted_words[:10]:
    print(word, ":", count)

print("\nPart D")
keyword = input("Enter keyword: ").lower()
found = False
for page_number, text in enumerate(pages, start=1):
    sentences = re.split(r"[.!?]", text)
    for sentence in sentences:
        if keyword in sentence.lower():
            print("Page", page_number, ":", sentence.strip())
            found = True
if found == False:
    print("Keyword not found.")
file.close()

Part A
Total pages extracted: 57

Part B
Total token count: 9140

Part C
Top 10 most frequent words:
const : 211
import : 132
gemini : 130
final : 92
dawakhana : 74
wahab : 73
color : 67
google : 65
page : 62
patient : 61

Part D
Page 1 : UNIVERSAL CHAT EXPORTER
Wahab Dawakhana - Google Gemini
Gemini  |  September 15, 2026  |  32 messages
YOU
YousaidFlutterMulti-Platform PatientEntryManagement App
I want to build a Flutter application that suppo
Page 1 : 
Page 1 : 
Page 1 : Flutter Multi-Platform Patient Entry Management App  I want to build a Flutter application that supports:  -
Mobile (Android & iOS)  - Web  - Desktop (Windows/macOS/Linux)  The app is intended for a doctor who
currently manages patient information manually in registers
Page 1 : The goal is to digitize and simplify patient record
management using a clean, fast, and user-friendly application
Page 1 : I will also provide UI design inspirations
separately
Page 1 : -----  Core Requirements  App Structure  The app will co

Task 2: Structured Page Metadata Extraction and CSV Export

In [4]:
import PyPDF2
import csv

print("Part A")
file = open("Wahab Dawakhana.pdf", "rb")
reader = PyPDF2.PdfReader(file)
pages_data = []
for page_number, page in enumerate(reader.pages, start=1):
    text = page.extract_text()
    if text is None:
        text = ""
    words = text.split()
    page_info = {
        "page_number": page_number,
        "raw_text": text,
        "word_count": len(words),
        "char_count": len(text.replace(" ", ""))
    }
    pages_data.append(page_info)
print("First 2 entries:")
for item in pages_data[:2]:
    print(item)

print("\nPart B")
highest_page = max(pages_data, key=lambda item: item["word_count"])
non_zero_pages = []
for item in pages_data:
    if item["word_count"] > 0:
        non_zero_pages.append(item)
lowest_page = min(non_zero_pages, key=lambda item: item["word_count"])
total_words = 0
for item in pages_data:
    total_words += item["word_count"]
average = round(total_words / len(pages_data), 2)

print("Page with highest word count:", highest_page["page_number"])
print("Word count:", highest_page["word_count"])
print(highest_page["raw_text"])
print("Page with lowest non-zero word count:", lowest_page["page_number"])
print("Word count:", lowest_page["word_count"])
print("Average word count:", average)

print("\nPart C")
class PhraseNotFoundError(Exception):
    pass
phrase = input("Enter search phrase: ").lower()
found = False
try:
    for item in pages_data:
        count = item["raw_text"].lower().count(phrase)
        if count > 0:
            print("Page", item["page_number"], "- Occurrences:", count)
            found = True
    if found == False:
        raise PhraseNotFoundError("Phrase not found in PDF.")
except PhraseNotFoundError as error:
    print(error)

print("\nPart D")
with open("pdf_summary.csv", "w", newline="") as csv_file:
    writer = csv.writer(csv_file)
    writer.writerow(["page_number", "word_count", "char_count"])
    for item in pages_data:
        writer.writerow([
            item["page_number"],
            item["word_count"],
            item["char_count"]
        ])
csv_file = open("pdf_summary.csv", "r")
reader = csv.reader(csv_file)
print("CSV contents:")
for row in reader:
    print(row)
csv_file.close()
file.close()

Part A
First 2 entries:
{'page_number': 1, 'raw_text': 'UNIVERSAL CHAT EXPORTER\nWahab Dawakhana - Google Gemini\nGemini  |  September 15, 2026  |  32 messages\nYOU\nYousaidFlutterMulti-Platform PatientEntryManagement App\nI want to build a Flutter application that suppo...\n Flutter Multi-Platform Patient Entry Management App  I want to build a Flutter application that supports:  -\nMobile (Android & iOS)  - Web  - Desktop (Windows/macOS/Linux)  The app is intended for a doctor who\ncurrently manages patient information manually in registers. The goal is to digitize and simplify patient record\nmanagement using a clean, fast, and user-friendly application.  I will also provide UI design inspirations\nseparately.  -----  Core Requirements  App Structure  The app will contain 2 main tabs/pages:  1. People  2.\nSettings  -----  1. People Page  This is the main section where patient entries are managed.  A. Patient List\nView  The People page should display a list of patient entries.  Eac

Task 3: PDF Text Preprocessing Pipeline for NLP

In [5]:
import PyPDF2
import re
import nltk
from nltk.corpus import stopwords
nltk.download("stopwords")

print("Part A")
file = open("Wahab Dawakhana.pdf", "rb")
reader = PyPDF2.PdfReader(file)
text = ""
for page in reader.pages:
    page_text = page.extract_text()
    if page_text is None:
        page_text = ""
    text += page_text + " "
before_count = len(text)
text = re.sub(r"http\S+|www\S+", "", text)
text = re.sub(r"\d+", "", text)
text = re.sub(r"[^A-Za-z. ]", "", text)
text = re.sub(r"\s+", " ", text)
after_count = len(text)
print("Character count before cleaning:", before_count)
print("Character count after cleaning:", after_count)

print("\nPart B")
text = text.lower()
sentences_before = re.split(r"[.!?]", text)
print("Sentence count before removing empty strings:", len(sentences_before))
sentences = []
for sentence in sentences_before:
    sentence = sentence.strip()
    if sentence != "":
        sentences.append(sentence)
print("Sentence count after removing empty strings:", len(sentences))

print("\nPart C")
stop_words = set(stopwords.words("english"))
filtered_sentences = []
print("Sentence count before filtering:", len(sentences))
for sentence in sentences:
    words = sentence.split()
    tokens = []
    for word in words:
        if word not in stop_words:
            tokens.append(word)
    if len(tokens) >= 5:
        filtered_sentences.append(" ".join(tokens))
print("Sentence count after filtering:", len(filtered_sentences))
print("Sample sentences:")
for sentence in filtered_sentences[:3]:
    print(sentence)

print("\nPart D")
output = open("nlp_ready_output.txt", "w")
for sentence in filtered_sentences:
    output.write(sentence + "\n")
output.close()
output = open("nlp_ready_output.txt", "r")
lines = output.readlines()
print("Total lines written:", len(lines))
longest = max(lines, key=len).strip()
shortest = min(lines, key=len).strip()
print("Longest sentence:")
print(longest)
print("Shortest sentence:")
print(shortest)
output.close()
file.close()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Omen\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


Part A
Character count before cleaning: 146304
Character count after cleaning: 115332

Part B
Sentence count before removing empty strings: 1850
Sentence count after removing empty strings: 1586

Part C
Sentence count before filtering: 1586
Sentence count after filtering: 569
Sample sentences:
universal chat exporterwahab dawakhana google geminigemini september messagesyouyousaidfluttermultiplatform patiententrymanagement appi want build flutter application suppo
flutter multiplatform patient entry management app want build flutter application supports mobile android ios web desktop windowsmacoslinux app intended doctor whocurrently manages patient information manually registers
goal digitize simplify patient recordmanagement using clean fast userfriendly application

Part D
Total lines written: 569
Longest sentence:
keyfeatures corecommunication realtime firebasechatmessages appearinstantlyonbothsidesusingfirestore livestreams livesigntotext translation camerarecognizes pslsignsandcon

Task 4: DOCX Paragraph Classification and Statistical Analysis

In [6]:
from docx import Document
from pprint import pprint

print("Part A")
document = Document("Lab 02 Data Acquisition for NLP.docx")
paragraphs_data = []
for paragraph in document.paragraphs:
    item = {
        "text": paragraph.text,
        "style": paragraph.style.name
    }
    paragraphs_data.append(item)
print("Total paragraph count:", len(paragraphs_data))
styles = set()
for item in paragraphs_data:
    styles.add(item["style"])
print("Unique style names:")
for style in styles:
    print(style)

print("\nPart B")
headings = []
body = []
other = []
for item in paragraphs_data:
    style = item["style"]
    text = item["text"]
    if style.startswith("Heading"):
        headings.append(item)
    elif style == "Normal" or "Body" in style:
        body.append(item)
    else:
        other.append(item)
print("Headings:", len(headings))
print("Body:", len(body))
print("Other/Empty:", len(other))

print("\nPart C")
body_info = []
for item in body:
    words = item["text"].split()
    if len(words) > 5:
        info = {
            "text": item["text"],
            "word_count": len(words),
            "char_count": len(item["text"].replace(" ", ""))
        }
        body_info.append(info)
highest = max(body_info, key=lambda item: item["word_count"])
lowest = min(body_info, key=lambda item: item["word_count"])
total = 0
for item in body_info:
    total += item["word_count"]
mean = round(total / len(body_info), 2)
print("Highest word count paragraph:")
print(highest["text"])
print("Word count:", highest["word_count"])
print("Character count:", highest["char_count"])
print("\nLowest word count paragraph above 5 words:")
print(lowest["text"])
print("Word count:", lowest["word_count"])
print("Character count:", lowest["char_count"])
print("\nMean word count:", mean)

print("\nPart D")
style_dict = {}
for item in paragraphs_data:
    style = item["style"]
    if style not in style_dict:
        style_dict[style] = {
            "count": 0,
            "texts": []
        }
    style_dict[style]["count"] += 1
    style_dict[style]["texts"].append(item["text"])
sorted_styles = dict(
    sorted(
        style_dict.items(),
        key=lambda item: item[1]["count"],
        reverse=True
    )
)
result = {}
for style, data in sorted_styles.items():
    if data["count"] > 2:
        result[style] = {
            "count": data["count"],
            "texts": data["texts"][:2]
        }
    else:
        result[style] = data
pprint(result)

Part A
Total paragraph count: 41
Unique style names:
Heading 2
pw-post-body-paragraph
List Paragraph
Normal

Part B
Headings: 6
Body: 27
Other/Empty: 8

Part C
Highest word count paragraph:
Transfer learning is a popular technique used in NLP that involves using pre-trained models to solve new tasks. Pre-trained models like BERT and GPT-2 have been trained on massive amounts of data and can be fine-tuned to solve new tasks with limited data. By leveraging transfer learning, we can save time and resources required to train models from scratch.
Word count: 59
Character count: 300

Lowest word count paragraph above 5 words:
Audio: Speech-to-text tools can be used to convert audio into machine-readable text.
Word count: 12
Character count: 73

Mean word count: 28.89

Part D
{'Heading 2': {'count': 6,
               'texts': ['Introduction to Data Acquisition for NLP',
                         'Three Types of Data Acquisition and Their '
                         'Challenges']},
 'List Parag

Task 5: DOCX Table Extraction and Pandas Integration

In [9]:
from docx import Document
import pandas as pd

print("Part A")
doc = Document("ORAS - Sufyan, Sabeen - Software Scope Document - Final.docx")
tables_data = []
for table in doc.tables:
    table_data = []
    for row in table.rows:
        row_data = []
        for cell in row.cells:
            row_data.append(cell.text)
        table_data.append(row_data)
    tables_data.append(table_data)
print("Total number of tables:", len(tables_data))
for index, table in enumerate(tables_data, start=1):
    rows = len(table)
    columns = len(table[0]) if rows > 0 else 0
    print("Table", index, "dimensions:", rows, "x", columns)

print("\nPart B")
if len(tables_data) > 0:
    first_table = tables_data[0]
    headers = first_table[0]
    data = first_table[1:]
    dataframe = pd.DataFrame(data, columns=headers)
    print("DataFrame:")
    print(dataframe)
    print("\nEmpty or None cells per column:")
    for column in dataframe.columns:
        count = 0
        for value in dataframe[column]:
            if value is None or str(value).strip() == "":
                count += 1
        print(column, ":", count)
else:
    print("No table found in the document.")

print("\nPart C")
pre_table_paragraphs = []
for element in doc.element.body:
    if element.tag.endswith("tbl"):
        break
    if element.tag.endswith("p"):
        text = "".join(element.itertext()).strip()
        pre_table_paragraphs.append(text)
print("Number of pre-table paragraphs:", len(pre_table_paragraphs))
for paragraph in pre_table_paragraphs:
    print(paragraph)

print("\nPart D")
output = open("docx_extracted.txt", "w", encoding="utf-8")
output.write("--- PRE-TABLE PARAGRAPHS ---\n")
for paragraph in pre_table_paragraphs:
    output.write(paragraph + "\n")
output.write("\n--- TABLE DATA ---\n")
for row in first_table:
    output.write(" | ".join(row) + "\n")
output.close()
print("Data written to docx_extracted.txt")
output = open("docx_extracted.txt", "r", encoding="utf-8")
print("\nFile contents:")
for line in output:
    print(line, end="")
output.close()

Part A
Total number of tables: 6
Table 1 dimensions: 7 x 3
Table 2 dimensions: 26 x 4
Table 3 dimensions: 2 x 2
Table 4 dimensions: 3 x 3
Table 5 dimensions: 28 x 3
Table 6 dimensions: 35 x 4

Part B
DataFrame:
           Application Name  \
0             Touch Surgery   
1                   Theator   
2             Standard PACS   
3  EndoNet (2017) [1] [15]    
4         TeCNO (2020) [5]    
5   Trans-SVNet (2021) [6]    

                                           Weakness   \
0  Relies entirely on pre-designed virtual simula...   
1  Closed enterprise platform with limited indivi...   
2  Treats surgical videos as passive, static file...   
3  Frame-based analysis with limited temporal rea...   
4  Focuses exclusively on workflow phase recognit...   
5  Research-focused implementation without practi...   

                           Proposed Project Solution  
0  ORAS extracts clinical insights directly from ...  
1  ORAS provides individual users with personaliz...  
2  ORAS conve

Task 6: Multi-Document Comparison, Deduplication, and Frequency Report

In [11]:
from docx import Document
import re
from collections import Counter

print("Part A")
document_A = Document("ORAS - Sufyan, Sabeen - Software Scope Document - Final.docx")
document_B = Document("Lab 02 Data Acquisition for NLP.docx")
headings_A = []
headings_B = []
for paragraph in document_A.paragraphs:
    if paragraph.style.name.startswith("Heading"):
        headings_A.append(paragraph.text.strip())
for paragraph in document_B.paragraphs:
    if paragraph.style.name.startswith("Heading"):
        headings_B.append(paragraph.text.strip())
set_A = set(headings_A)
set_B = set(headings_B)
common_headings = set_A.intersection(set_B)
only_A = set_A.difference(set_B)
only_B = set_B.difference(set_A)
print("Headings common to both documents:", common_headings)
print("Headings only in Document A:", only_A)
print("Headings only in Document B:", only_B)

print("\nPart B")
body_A = []
body_B = []
for paragraph in document_A.paragraphs:
    if paragraph.style.name == "Normal":
        body_A.append(paragraph.text)
for paragraph in document_B.paragraphs:
    if paragraph.style.name == "Normal":
        body_B.append(paragraph.text)
combined_body = body_A + body_B
print("Total paragraphs before deduplication:", len(combined_body))
unique_body = []
seen = set()
for paragraph in combined_body:
    clean_text = paragraph.strip().lower()

    if clean_text not in seen:
        seen.add(clean_text)
        unique_body.append(paragraph)
print("Total paragraphs after deduplication:", len(unique_body))

print("\nPart C")
stopwords = {
    "the", "is", "a", "an", "and", "of", "to", "in",
    "on", "for", "with", "that", "this", "it", "as",
    "by", "from", "or", "be", "are"
}
word_frequency = Counter()
for paragraph in unique_body:
    text = paragraph.lower()
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    words = text.split()
    for word in words:
        if word not in stopwords:
            word_frequency[word] += 1
print("Top 20 most frequent words:")
for word, count in word_frequency.most_common(20):
    print(word, ":", count)

print("\nPart D")
total_headings = len(headings_A) + len(headings_B)
total_unique_body = len(unique_body)
vocabulary_size = len(word_frequency)
total_words = 0
for paragraph in unique_body:
    text = re.sub(r"[^a-zA-Z\s]", "", paragraph.lower())
    words = text.split()
    total_words += len(words)
if len(unique_body) > 0:
    average_length = round(total_words / len(unique_body), 2)
else:
    average_length = 0
if len(unique_body) > 0:
    longest_paragraph = max(
        unique_body,
        key=lambda paragraph: len(
            re.sub(r"[^a-zA-Z\s]", "", paragraph.lower()).split()
        )
    )
    longest_words = len(
        re.sub(r"[^a-zA-Z\s]", "", longest_paragraph.lower()).split()
    )
else:
    longest_paragraph = ""
    longest_words = 0
print("Total headings across both documents:", total_headings)
print("Total unique body paragraphs:", total_unique_body)
print("Total unique word types:", vocabulary_size)
print("Average paragraph length in words:", average_length)
print("Longest paragraph word count:", longest_words)
print("Longest paragraph:")
print(longest_paragraph)

Part A
Headings common to both documents: set()
Headings only in Document A: {'', 'Objectives', 'Mockups', 'Vision Statement', 'WBS and Gantt Chart', '7.4 - Module 4: Surgical Tool Detection & Tracking Engine', '7.6 - Module 6: Intelligent Event Indexing & Time-Log Analytics', 'Modules', 'Scope', '7.3 - Module 3: Cloud-Native Web Ingestion Dashboard', 'Introduction', 'Plagiarism Report', 'Abstract', 'Tools and Technologies', '7.2 - Module 2: Hybrid Mobile Video Ingestion & Permissions', 'References', '7.5 - Module 5: Temporal Operative Phase Recognition Engine [2]', 'System Limitations/Constraints', 'Related System Analysis/Literature Review', 'Module based Work Division', '7.10 - Module 10: Multi-Tier Commercial Subscription & Billing Management', 'Project Stakeholders and Roles', '7.7 - Module 7: Interactive Review, Search & Jump-to Navigation', '7.8 - Module 8: Automated Post-Op Reporting & Storage Sync', 'Problem Solution/Objectives of the Proposed System', 'Problem Statement', '7.

Task 7: Multi-Endpoint API Data Joining and Analysis

In [13]:
import requests

print("Part A")
users_url = "https://jsonplaceholder.typicode.com/users"
posts_url = "https://jsonplaceholder.typicode.com/posts"
users_response = requests.get(users_url)
posts_response = requests.get(posts_url)
if users_response.status_code != 200:
    raise ConnectionError("Users request failed with status code: " + str(users_response.status_code))
if posts_response.status_code != 200:
    raise ConnectionError("Posts request failed with status code: " + str(posts_response.status_code))
users = users_response.json()
posts = posts_response.json()
print("Users status code:", users_response.status_code)
print("Posts status code:", posts_response.status_code)
print("Total users:", len(users))
print("Total posts:", len(posts))

print("\nPart B")
joined_data = {}
for user in users:
    user_name = user["name"]
    user_posts = []
    for post in posts:
        if post["userId"] == user["id"]:
            user_posts.append(post["title"])
    joined_data[user_name] = {
        "email": user["email"],
        "post_count": len(user_posts),
        "titles": user_posts
    }
print("First 3 users:")
count = 0
for name, data in joined_data.items():
    print(name, ":", data)
    count += 1
    if count == 3:
        break

print("\nPart C")
most_posts_user = max(
    users,
    key=lambda user: joined_data[user["name"]]["post_count"]
)
user_name = most_posts_user["name"]
user_data = joined_data[user_name]
print("Full name:", user_name)
print("Email address:", most_posts_user["email"])
print("Company name:", most_posts_user["company"]["name"])
print("Post titles:")
for number, title in enumerate(user_data["titles"], start=1):
    print(number, ".", title)

print("\nPart D")
for user in users:
    company_bs = user["company"]["bs"].lower()
    city = user["address"]["city"]
    if "synergies" in company_bs or len(city) > 8:
        name = user["name"]
        post_count = joined_data[name]["post_count"]
        print("Name:", name)
        print("City:", city)
        print("Post count:", post_count)
        print()

Part A
Users status code: 200
Posts status code: 200
Total users: 10
Total posts: 100

Part B
First 3 users:
Leanne Graham : {'email': 'Sincere@april.biz', 'post_count': 10, 'titles': ['sunt aut facere repellat provident occaecati excepturi optio reprehenderit', 'qui est esse', 'ea molestias quasi exercitationem repellat qui ipsa sit aut', 'eum et est occaecati', 'nesciunt quas odio', 'dolorem eum magni eos aperiam quia', 'magnam facilis autem', 'dolorem dolore est ipsam', 'nesciunt iure omnis dolorem tempora et accusantium', 'optio molestias id quia eum']}
Ervin Howell : {'email': 'Shanna@melissa.tv', 'post_count': 10, 'titles': ['et ea vero quia laudantium autem', 'in quibusdam tempore odit est dolorem', 'dolorum ut in voluptas mollitia et saepe quo animi', 'voluptatem eligendi optio', 'eveniet quod temporibus', 'sint suscipit perspiciatis velit dolorum rerum ipsa laboriosam odio', 'fugit voluptas sed molestias voluptatem provident', 'voluptate et itaque vero tempora molestiae', 'adi

Task 8: To-Do Completion Rate Analysis Across Users

In [15]:
import requests

print("Part A")
todos_url = "https://jsonplaceholder.typicode.com/todos"
response = requests.get(todos_url)
if response.status_code != 200:
    print("Error: Request failed with status code", response.status_code)
    exit()
todos = response.json()
if type(todos) != list:
    print("Error: The response is not a list.")
    exit()
if len(todos) == 0:
    print("Error: The todo list is empty.")
    exit()
print("Total todo records retrieved:", len(todos))

print("\nPart B")
user_todos = {}
for todo in todos:
    user_id = todo["userId"]
    if user_id not in user_todos:
        user_todos[user_id] = {
            "total": 0,
            "completed": 0,
            "incomplete": 0
        }
    user_todos[user_id]["total"] += 1
    if todo["completed"] == True:
        user_todos[user_id]["completed"] += 1
    else:
        user_todos[user_id]["incomplete"] += 1
for user_id in sorted(user_todos):
    data = user_todos[user_id]
    check = data["completed"] + data["incomplete"] == data["total"]
    print("User", user_id, ":", data)
    print("Completed + Incomplete == Total:", check)

print("\nPart C")
completion_rates = {}
for user_id, data in user_todos.items():
    rate = round((data["completed"] / data["total"]) * 100, 1)
    completion_rates[user_id] = rate
highest_user = max(completion_rates, key=completion_rates.get)
lowest_user = min(completion_rates, key=completion_rates.get)
total_rate = 0
for rate in completion_rates.values():
    total_rate += rate
average_rate = round(total_rate / len(completion_rates), 1)
print("Highest completion rate:")
print("User ID:", highest_user)
print("Completion rate:", completion_rates[highest_user], "%")
print("Lowest completion rate:")
print("User ID:", lowest_user)
print("Completion rate:", completion_rates[lowest_user], "%")
print("Overall average completion rate:", average_rate, "%")

print("\nPart D")
users_url = "https://jsonplaceholder.typicode.com/users"
users_response = requests.get(users_url)
if users_response.status_code != 200:
    print("Error: Users request failed with status code", users_response.status_code)
    exit()
users = users_response.json()
user_names = {}
for user in users:
    user_names[user["id"]] = user["name"]
sorted_users = sorted(
    completion_rates.items(),
    key=lambda item: item[1],
    reverse=True
)
print(f'{"Rank":<6}{"Name":<25}{"Total Tasks":<15}{"Completed":<12}{"Completion Rate %":<20}')
print("-" * 78)
for rank, (user_id, rate) in enumerate(sorted_users, start=1):
    name = user_names[user_id]
    total = user_todos[user_id]["total"]
    completed = user_todos[user_id]["completed"]
    print(f"{rank:<6}{name:<25}{total:<15}{completed:<12}{rate:<20.1f}")

Part A
Total todo records retrieved: 200

Part B
User 1 : {'total': 20, 'completed': 11, 'incomplete': 9}
Completed + Incomplete == Total: True
User 2 : {'total': 20, 'completed': 8, 'incomplete': 12}
Completed + Incomplete == Total: True
User 3 : {'total': 20, 'completed': 7, 'incomplete': 13}
Completed + Incomplete == Total: True
User 4 : {'total': 20, 'completed': 6, 'incomplete': 14}
Completed + Incomplete == Total: True
User 5 : {'total': 20, 'completed': 12, 'incomplete': 8}
Completed + Incomplete == Total: True
User 6 : {'total': 20, 'completed': 6, 'incomplete': 14}
Completed + Incomplete == Total: True
User 7 : {'total': 20, 'completed': 9, 'incomplete': 11}
Completed + Incomplete == Total: True
User 8 : {'total': 20, 'completed': 11, 'incomplete': 9}
Completed + Incomplete == Total: True
User 9 : {'total': 20, 'completed': 8, 'incomplete': 12}
Completed + Incomplete == Total: True
User 10 : {'total': 20, 'completed': 12, 'incomplete': 8}
Completed + Incomplete == Total: True


Task 9: Nested JSON Flattening, DataFrame Operations, and Comment Analysis

In [17]:
import requests
import pandas as pd
import math

print("Part A")
url = "https://jsonplaceholder.typicode.com/users"
response = requests.get(url)
if response.status_code != 200:
    print("Error: Request failed with status code", response.status_code)
    exit()
users = response.json()
def flatten_user(user):
    return {
        "id": user["id"],
        "name": user["name"],
        "username": user["username"],
        "email": user["email"],
        "city": user["address"]["city"],
        "zipcode": user["address"]["zipcode"],
        "lat": user["address"]["geo"]["lat"],
        "lng": user["address"]["geo"]["lng"],
        "company_name": user["company"]["name"],
        "company_catchphrase": user["company"]["catchPhrase"]
    }
flattened_users = []
for user in users:
    flattened_users.append(flatten_user(user))
print("Flattened user list:")
print(flattened_users)

print("\nPart B")
df = pd.DataFrame(flattened_users)
df["lat"] = df["lat"].astype(float)
df["lng"] = df["lng"].astype(float)
df["distance_from_origin"] = df.apply(
    lambda row: math.sqrt(row["lat"] ** 2 + row["lng"] ** 2),
    axis=1
)
print("DataFrame:")
print(df)
greatest_distance = df.loc[df["distance_from_origin"].idxmax()]
print("\nUser with greatest distance from origin:")
print(greatest_distance)

print("\nPart C")
northern_users = df[df["lat"] > 0]
southern_users = df[df["lat"] < 0]
print("Northern Hemisphere users:")
for name in northern_users["name"]:
    print(name)
print("\nSouthern Hemisphere users:")
for name in southern_users["name"]:
    print(name)
northern_average = northern_users["distance_from_origin"].mean()
southern_average = southern_users["distance_from_origin"].mean()
print("\nNorthern Hemisphere average distance:", round(northern_average, 2))
print("Southern Hemisphere average distance:", round(southern_average, 2))
if northern_average > southern_average:
    print("Northern Hemisphere has the higher average distance.")
else:
    print("Southern Hemisphere has the higher average distance.")

print("\nPart D")
posts_url = "https://jsonplaceholder.typicode.com/posts"
comments_url = "https://jsonplaceholder.typicode.com/comments"
posts_response = requests.get(posts_url)
comments_response = requests.get(comments_url)
if posts_response.status_code != 200:
    print("Error: Posts request failed with status code", posts_response.status_code)
    exit()
if comments_response.status_code != 200:
    print("Error: Comments request failed with status code", comments_response.status_code)
    exit()
posts = posts_response.json()
comments = comments_response.json()
comment_counts = {}
for post in posts:
    comment_counts[post["id"]] = 0
for comment in comments:
    post_id = comment["postId"]
    if post_id in comment_counts:
        comment_counts[post_id] += 1
post_data = []
for post in posts:
    post_id = post["id"]
    title = post["title"][:40]
    count = comment_counts[post_id]
    post_data.append((post_id, title, count))
post_data = sorted(post_data, key=lambda item: item[2], reverse=True)
print("Top 5 most-commented posts:")
for item in post_data[:5]:
    print(item)
print("\nBottom 5 least-commented posts:")
for item in post_data[-5:]:
    print(item)

Part A
Flattened user list:
[{'id': 1, 'name': 'Leanne Graham', 'username': 'Bret', 'email': 'Sincere@april.biz', 'city': 'Gwenborough', 'zipcode': '92998-3874', 'lat': '-37.3159', 'lng': '81.1496', 'company_name': 'Romaguera-Crona', 'company_catchphrase': 'Multi-layered client-server neural-net'}, {'id': 2, 'name': 'Ervin Howell', 'username': 'Antonette', 'email': 'Shanna@melissa.tv', 'city': 'Wisokyburgh', 'zipcode': '90566-7771', 'lat': '-43.9509', 'lng': '-34.4618', 'company_name': 'Deckow-Crist', 'company_catchphrase': 'Proactive didactic contingency'}, {'id': 3, 'name': 'Clementine Bauch', 'username': 'Samantha', 'email': 'Nathan@yesenia.net', 'city': 'McKenziehaven', 'zipcode': '59590-4157', 'lat': '-68.6102', 'lng': '-47.0653', 'company_name': 'Romaguera-Jacobson', 'company_catchphrase': 'Face to face bifurcated interface'}, {'id': 4, 'name': 'Patricia Lebsack', 'username': 'Karianne', 'email': 'Julianne.OConner@kory.org', 'city': 'South Elvis', 'zipcode': '53919-4257', 'lat': 